In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from dotenv import load_dotenv
import pandas as pd
from langchain_core.documents import Document
from torch.nn.functional import embedding


In [2]:

load_dotenv()
books =  pd.read_csv("books_cleaned.csv")
books

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,age of books,title_and_subtitle,tagged_description
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,361.0,22.0,Gilead,9780002005883:A NOVEL THAT READERS and critics...
1,9780002261982,0002261987,Spider's Web,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,5164.0,26.0,Spider's Web:A Novel,9780002261982:A new 'Christie for Christmas' -...
2,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"A memorable, mesmerizing heroine Jennifer -- b...",1993.0,3.93,512.0,29532.0,33.0,Rage of angels,"9780006178736:A memorable, mesmerizing heroine..."
3,9780006280897,0006280897,The Four Loves,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...,Lewis' work on the nature of love divides love...,2002.0,4.15,170.0,33684.0,24.0,The Four Loves,9780006280897:Lewis' work on the nature of lov...
4,9780006280934,0006280935,The Problem of Pain,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=Kk-uV...,"""In The Problem of Pain, C.S. Lewis, one of th...",2002.0,4.09,176.0,37569.0,24.0,The Problem of Pain,"9780006280934:""In The Problem of Pain, C.S. Le..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5192,9788172235222,8172235224,Mistaken Identity,Nayantara Sahgal,Indic fiction (English),http://books.google.com/books/content?id=q-tKP...,On A Train Journey Home To North India After L...,2003.0,2.93,324.0,0.0,23.0,Mistaken Identity,9788172235222:On A Train Journey Home To North...
5193,9788173031014,8173031010,Journey to the East,Hermann Hesse,Adventure stories,http://books.google.com/books/content?id=rq6JP...,This book tells the tale of a man who goes on ...,2002.0,3.70,175.0,24.0,24.0,Journey to the East,9788173031014:This book tells the tale of a ma...
5194,9788179921623,817992162X,The Monk Who Sold His Ferrari: A Fable About F...,Robin Sharma,Health & Fitness,http://books.google.com/books/content?id=c_7mf...,"Wisdom to Create a Life of Passion, Purpose, a...",2003.0,3.82,198.0,1568.0,23.0,The Monk Who Sold His Ferrari: A Fable About F...,9788179921623:Wisdom to Create a Life of Passi...
5195,9788185300535,8185300534,I Am that,Sri Nisargadatta Maharaj;Sudhakar S. Dikshit,Philosophy,http://books.google.com/books/content?id=Fv_JP...,This collection of the timeless teachings of o...,1999.0,4.51,531.0,104.0,27.0,I Am that:Talks with Sri Nisargadatta Maharaj,9788185300535:This collection of the timeless ...


In [3]:
books["tagged_description"]

0       9780002005883:A NOVEL THAT READERS and critics...
1       9780002261982:A new 'Christie for Christmas' -...
2       9780006178736:A memorable, mesmerizing heroine...
3       9780006280897:Lewis' work on the nature of lov...
4       9780006280934:"In The Problem of Pain, C.S. Le...
                              ...                        
5192    9788172235222:On A Train Journey Home To North...
5193    9788173031014:This book tells the tale of a ma...
5194    9788179921623:Wisdom to Create a Life of Passi...
5195    9788185300535:This collection of the timeless ...
5196    9789027712059:Since the three volume edition o...
Name: tagged_description, Length: 5197, dtype: str

In [4]:
books["tagged_description"].to_csv("tagged_description.txt" , index=False , header = False)

In [5]:
#raw_documents = TextLoader("tagged_description.txt", encoding="utf-8").load()
#text_splitter = CharacterTextSplitter(chunk_size=10000 , chunk_overlap=0 , separator="\n")
#documents = text_splitter.split_documents(raw_documents)
#did not use the character text spliter since the document is already structured as a book per line also the tutorial i follow uses chunk_size=0 which is not suported by the langchain im using in this code
from langchain_core.documents import Document

with open("tagged_description.txt", "r", encoding="utf-8") as f:
    documents = [
        Document(
            page_content=line.strip(),
            metadata={"source": "tagged_description.txt"}
        )
        for line in f
        if line.strip()
    ]

print(len(documents))
print(documents[0])
print(documents[1])

5197
page_content='"9780002005883:A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives. John Ames is a preacher, the son of a preacher and the grandson (both maternal and paternal) of preachers. It’s 1956 in Gilead, Iowa, towards the end of the Reverend Ames’s life, and he is absorbed in recording his family’s story, a legacy for the young son he will never see grow up. Haunted by his grandfather’s presence, John tells of the rift between his grandfather and his father: the elder, an angry visionary who fought for the abolitionist cause, and his son, an ardent pacifist. He is troubled, too, by his prodigal namesake, Jack (John Ames) Boughton, his best friend’s lost son who returns to Gilead searching for forgiveness and redemption. Told in John Ames’s joyous, rambling voice that finds beauty, humour and truth in the smallest of life’s details, Gilead is a song of celebration and acceptance of th

In [6]:
documents[0]

Document(metadata={'source': 'tagged_description.txt'}, page_content='"9780002005883:A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives. John Ames is a preacher, the son of a preacher and the grandson (both maternal and paternal) of preachers. It’s 1956 in Gilead, Iowa, towards the end of the Reverend Ames’s life, and he is absorbed in recording his family’s story, a legacy for the young son he will never see grow up. Haunted by his grandfather’s presence, John tells of the rift between his grandfather and his father: the elder, an angry visionary who fought for the abolitionist cause, and his son, an ardent pacifist. He is troubled, too, by his prodigal namesake, Jack (John Ames) Boughton, his best friend’s lost son who returns to Gilead searching for forgiveness and redemption. Told in John Ames’s joyous, rambling voice that finds beauty, humour and truth in the smallest of life’s details, G

In [8]:

print(len(documents))

print(documents[0])
print(documents[1])
print(documents[2])

5197
page_content='"9780002005883:A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives. John Ames is a preacher, the son of a preacher and the grandson (both maternal and paternal) of preachers. It’s 1956 in Gilead, Iowa, towards the end of the Reverend Ames’s life, and he is absorbed in recording his family’s story, a legacy for the young son he will never see grow up. Haunted by his grandfather’s presence, John tells of the rift between his grandfather and his father: the elder, an angry visionary who fought for the abolitionist cause, and his son, an ardent pacifist. He is troubled, too, by his prodigal namesake, Jack (John Ames) Boughton, his best friend’s lost son who returns to Gilead searching for forgiveness and redemption. Told in John Ames’s joyous, rambling voice that finds beauty, humour and truth in the smallest of life’s details, Gilead is a song of celebration and acceptance of th

In [9]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

db_books = Chroma.from_documents(documents, embedding=embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [10]:
query = "A book to teach children about nature"
docs = db_books.similarity_search(query, k=10)
docs


[Document(id='24fd3162-bd9d-461f-ac59-03e15577d3ba', metadata={'source': 'tagged_description.txt'}, page_content='"9780786808069:Children will discover the exciting world of their own backyard in this introduction to familiar animals from cats and dogs to bugs and frogs. The combination of photographs, illustrations, and fun facts make this an accessible and delightful learning experience."'),
 Document(id='ac2d99bd-1fb7-4c93-bfa9-482efd94feec', metadata={'source': 'tagged_description.txt'}, page_content='"9780786808380:Introduce your babies to birds, cats, dogs, and babies through fine art, illustration, and photographs. These books are a rare opportunity to expose little ones to a range of images on a single subject, from simple child\'s drawings and abstract art to playful photos. A brief text accompanies each image, introducing the baby to some basic -- and sometimes playful -- information about the subjects."'),
 Document(id='49e0e6ca-a8f5-48ad-85b9-bcdfe81fb874', metadata={'sourc

In [13]:
books[books ["isbn13"]==int(docs[0].page_content.split()[0].strip('"').split(":")[0])]

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,age of books,title_and_subtitle,tagged_description
3747,9780786808069,0786808063,Baby Einstein: Neighborhood Animals,Marilyn Singer;Julie Aigner-Clark,Juvenile Fiction,http://books.google.com/books/content?id=X9a4P...,Children will discover the exciting world of t...,2001.0,3.89,16.0,180.0,25.0,Baby Einstein: Neighborhood Animals,9780786808069:Children will discover the excit...


In [28]:
def retrieve_sementic_recommendations(
        query: str,
        top_k: int = 10,

) -> pd.DataFrame:
    recs =  db_books.similarity_search(query, k = 50)
    books_list = []
    for i in range(0, len(recs)):
        isbn = int(
            recs[i].page_content.strip('"').split(":")[0])
        books_list.append(isbn)
    #return books[books["isbn13"].isin(books_list)].head(top_k)
    return(books.set_index("isbn13").loc[books_list].head(top_k).reset_index())

In [25]:
retrieve_sementic_recommendations("A book to teach children about nature")

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,age of books,title_and_subtitle,tagged_description
0,9780786808069,0786808063,Baby Einstein: Neighborhood Animals,Marilyn Singer;Julie Aigner-Clark,Juvenile Fiction,http://books.google.com/books/content?id=X9a4P...,Children will discover the exciting world of t...,2001.0,3.89,16.0,180.0,25.0,Baby Einstein: Neighborhood Animals,9780786808069:Children will discover the excit...
1,9780786808380,0786808381,Baby Einstein: Babies,Julie Aigner-Clark,Juvenile Fiction,http://books.google.com/books/content?id=jv4NA...,"Introduce your babies to birds, cats, dogs, an...",2002.0,4.03,20.0,29.0,24.0,Baby Einstein: Babies,"9780786808380:Introduce your babies to birds, ..."
2,9780786808397,078680839X,Baby Einstein: Dogs,Julie Aigner-Clark,Juvenile Fiction,http://books.google.com/books/content?id=qut8t...,"Introduce your baby to birds, cats, dogs, and ...",2002.0,3.81,20.0,26.0,24.0,Baby Einstein: Dogs,"9780786808397:Introduce your baby to birds, ca..."
3,9780786808373,0786808373,Baby Einstein: Birds,Julie Aigner-Clark,Juvenile Fiction,http://books.google.com/books/content?id=0jxHP...,"Introducing your baby to birds, cats, dogs, an...",2002.0,3.78,20.0,9.0,24.0,Baby Einstein: Birds,"9780786808373:Introducing your baby to birds, ..."
4,9780060959036,0060959037,Prodigal Summer,Barbara Kingsolver,Fiction,http://books.google.com/books/content?id=06IwG...,Barbara Kingsolver's fifth novel is a hymn to ...,2001.0,4.00,444.0,85440.0,25.0,Prodigal Summer:A Novel,9780060959036:Barbara Kingsolver's fifth novel...
5,9780374522599,0374522596,The Control of Nature,John McPhee,Nature,http://books.google.com/books/content?id=p1qKQ...,The Control of Nature is John McPhee's bestsel...,1990.0,4.24,288.0,3365.0,36.0,The Control of Nature,9780374522599:The Control of Nature is John Mc...
6,9780064402453,0064402452,Racso and the Rats of NIMH,Jane Leslie Conly,Juvenile Fiction,http://books.google.com/books/content?id=MgoNv...,"‘Racso, a brash and boastful little rodent, is...",1988.0,3.76,288.0,3231.0,38.0,Racso and the Rats of NIMH,"9780064402453:‘Racso, a brash and boastful lit..."
7,9780786819119,0786819111,"Baby Einstein: Water, Water Everywhere","Disney Book Group,",Juvenile Fiction,http://books.google.com/books/content?id=tuAdA...,Charming illustrations and playful rhythmic ve...,2003.0,3.70,10.0,77.0,23.0,"Baby Einstein: Water, Water Everywhere",9780786819119:Charming illustrations and playf...
8,9780064403870,0064403874,"R-T, Margaret, and the Rats of NIMH",Jane Leslie Conly,Juvenile Fiction,http://books.google.com/books/content?id=WTHHH...,"When Margaret and her younger brother, Artie, ...",1991.0,3.52,272.0,631.0,35.0,"R-T, Margaret, and the Rats of NIMH",9780064403870:When Margaret and her younger br...
9,9780802431486,0802431488,The 10 Commandments of Parenting,H. Edwin Young,Religion,http://books.google.com/books/content?id=Q1Hxj...,Drawn from years of counseling and the author'...,2005.0,4.00,224.0,25.0,21.0,The 10 Commandments of Parenting:The Do's and ...,9780802431486:Drawn from years of counseling a...


In [30]:
retrieve_sementic_recommendations("A book that teaches math")

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,age of books,title_and_subtitle,tagged_description
0,9780747569138,0747569134,More Sideways Arithmetic from Wayside School,Louis Sachar,Arithmetic,http://books.google.com/books/content?id=EMj7H...,Welcome back to maths class at Wayside School....,2004.0,3.92,112.0,839.0,22.0,More Sideways Arithmetic from Wayside School,9780747569138:Welcome back to maths class at W...
1,9780385492249,0385492243,An Invisible Sign of My Own,Aimee Bender,Fiction,http://books.google.com/books/content?id=ZLRKD...,"Mona Gray, a young second grade teacher who or...",2001.0,3.68,256.0,3624.0,25.0,An Invisible Sign of My Own,"9780385492249:Mona Gray, a young second grade ..."
2,9780440241355,0440241359,The Rule of Four,Ian Caldwell;Dustin Thomason,Fiction,http://books.google.com/books/content?id=KAtIT...,Trying to decipher an ancient text that weaves...,2005.0,3.22,450.0,24001.0,21.0,The Rule of Four,9780440241355:Trying to decipher an ancient te...
3,9780870610639,0870610635,Summa Theologica,Thomas Aquinas,"Theology, Doctrinal",http://books.google.com/books/content?id=l9R4x...,Creating a summary of all human knowledge may ...,2000.0,4.10,3020.0,2673.0,26.0,Summa Theologica:Complete 5-Volume Set,9780870610639:Creating a summary of all human ...
4,9780099928409,009992840X,Ratner's Star,Don DeLillo,Life on other planets,http://books.google.com/books/content?id=GTJAN...,Billy Twillig has won the first Nobel Prize ev...,1976.0,3.49,448.0,1478.0,50.0,Ratner's Star,9780099928409:Billy Twillig has won the first ...
5,9780747569121,0747569126,Sideways Arithmetic from Wayside School,Louis Sachar,Arithmetic,http://books.google.com/books/content?id=-oViP...,These Sideways Arithmetic problems may look pu...,2004.0,3.87,96.0,3627.0,22.0,Sideways Arithmetic from Wayside School,9780747569121:These Sideways Arithmetic proble...
6,9780691050843,0691050848,The Nature of Space and Time,Stephen Hawking;Roger Penrose,Science,http://books.google.com/books/content?id=LstaQ...,Presents a series of lectures delivered in 199...,2000.0,4.09,152.0,945.0,26.0,The Nature of Space and Time,9780691050843:Presents a series of lectures de...
7,9780439569729,0439569729,SCHOLASTIC SUCCESS WITH 4TH GRADE(WORKBOOK),Terry Cooper,Education,http://books.google.com/books/content?id=b-EBH...,"416 bright, colorful pages that give kids prac...",2003.0,4.57,416.0,7.0,23.0,SCHOLASTIC SUCCESS WITH 4TH GRADE(WORKBOOK),"9780439569729:416 bright, colorful pages that ..."
8,9780131871656,013187165X,Astronomy,Eric Chaisson;Stephen McMillan,Mathematics,http://books.google.com/books/content?id=1O00A...,This introduction to astronomy features an exc...,2006.0,3.85,499.0,153.0,20.0,Astronomy:a beginner's guide to the universe,9780131871656:This introduction to astronomy f...
9,9780762419227,0762419229,God Created The Integers,Stephen Hawking,Mathematics,http://books.google.com/books/content?id=3zdFS...,Looks at landmark mathematical discoveries ove...,2005.0,4.06,1160.0,1650.0,21.0,God Created The Integers,9780762419227:Looks at landmark mathematical d...


In [32]:
retrieve_sementic_recommendations("a book that helps with mental health ")

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,age of books,title_and_subtitle,tagged_description
0,9780330346511,0330346512,An Unquiet Mind,Kay Redfield Jamison,Biography & Autobiography,http://books.google.com/books/content?id=adEqu...,"In this book, a world authority on manic-depre...",1997.0,4.05,240.0,297.0,29.0,An Unquiet Mind:A Memoir of Moods and Madness,"9780330346511:In this book, a world authority ..."
1,9780684854670,0684854678,The Noonday Demon,Andrew Solomon,Psychology,http://books.google.com/books/content?id=ZgbbA...,Winner of the National Book Award and a Pulitz...,2002.0,4.17,576.0,8749.0,24.0,The Noonday Demon:An Atlas of Depression,9780684854670:Winner of the National Book Awar...
2,9780451160317,0451160312,I Never Promised You a Rose Garden,Joanne Greenberg,Mental illness,http://books.google.com/books/content?id=I2R1x...,Chronicles the three-year battle of a mentally...,1989.0,3.86,288.0,23725.0,37.0,I Never Promised You a Rose Garden,9780451160317:Chronicles the three-year battle...
3,9780195135794,0195135792,Manic-Depressive Illness,Frederick K. Goodwin;Kay Redfield Jamison,Medical,http://books.google.com/books/content?id=hOHWE...,This long-awaited second edition of Manic-Depr...,2007.0,4.39,1262.0,103.0,19.0,Manic-Depressive Illness:Bipolar Disorders and...,9780195135794:This long-awaited second edition...
4,9781573229623,1573229628,Prozac Nation,Elizabeth Wurtzel,Biography & Autobiography,http://books.google.com/books/content?id=441Xv...,"A memoir of sex, drugs, and depression indicts...",2002.0,3.59,384.0,875.0,24.0,Prozac Nation:Young and Depressed in America,"9781573229623:A memoir of sex, drugs, and depr..."
5,9780345487421,0345487427,Feel the Fear-- and Do it Anyway,Susan J. Jeffers,Psychology,http://books.google.com/books/content?id=o2Wbc...,A psychotherapist shows how to identify the fe...,2007.0,4.03,217.0,474.0,19.0,Feel the Fear-- and Do it Anyway,9780345487421:A psychotherapist shows how to i...
6,9780452281325,0452281326,The Feeling Good Handbook,David D. Burns,Psychology,http://books.google.com/books/content?id=akVIQ...,"Discusses how to overcome fears, phobias, and ...",1999.0,4.00,729.0,4666.0,27.0,The Feeling Good Handbook,"9780452281325:Discusses how to overcome fears,..."
7,9780316578998,0316578991,The Black Veil,Rick Moody,Family & Relationships,http://books.google.com/books/content?id=AQC2Q...,The author weaves together past and present an...,2002.0,3.06,336.0,391.0,24.0,The Black Veil:A Memoir with Digressions,9780316578998:The author weaves together past ...
8,9780781731836,0781731836,Kaplan & Sadock's Synopsis of Psychiatry,Benjamin J. Sadock;Virginia A. Sadock,Medical,http://books.google.com/books/content?id=YX5mQ...,The best-selling general psychiatry text since...,2003.0,4.20,1500.0,236.0,23.0,Kaplan & Sadock's Synopsis of Psychiatry:Behav...,9780781731836:The best-selling general psychia...
9,9780684831831,068483183X,Touched With Fire,Kay Redfield Jamison,Psychology,http://books.google.com/books/content?id=265d7...,The definitive work on the profound and surpri...,1996.0,4.01,384.0,4087.0,30.0,Touched With Fire,9780684831831:The definitive work on the profo...
